# Full Pipeline Run Notebook

Run the **end-to-end LangGraph pipeline** across all chunks in a chapter via `graph.invoke()`.

**Rules:**
- Implementation lives in `src/` — this notebook is for execution and inspection only.
- Pipeline: `check_database` → `draft` → `judge` → `revise` (loop, max 3) → `save_to_db`

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import hashlib
from pathlib import Path
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table

load_dotenv()
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from note_taker.database import DatabaseManager
from note_taker.processing import parse_markdown_chunks
from note_taker.pipeline.graph import build_graph

DatabaseManager._instance = None
db = DatabaseManager('dev_notes.db')
db.ensure_database()
rprint('[green]Setup complete[/green]')

## Load + Chunk Chapter

In [ ]:
DATA_DIR = Path().resolve().parent / 'data' / 'raw' / 'building-applications-with'
target_file = DATA_DIR / 'chapter_005.md'
BOOK_CHAPTER = 'BuildingAIAgents:Chapter5'

chunks = parse_markdown_chunks(str(target_file))

table = Table(title=f'{target_file.name} — {len(chunks)} chunks')
table.add_column('Idx', style='dim', width=4)
table.add_column('Title', style='cyan')
table.add_column('Chars', justify='right', style='green')

for i, c in enumerate(chunks):
    table.add_row(f'{i:03d}', c['title'], str(len(c['content'])))

rprint(table)

## Run Full LangGraph Pipeline
Invoke the compiled graph on all chunks. The graph handles check_database → draft → judge → revise → save automatically.

In [ ]:
graph = build_graph()

skipped = 0
processed = 0
errors = []

for idx, chunk in enumerate(chunks):
    chunk_id = f'{BOOK_CHAPTER}:{idx:03d}'
    source_hash = hashlib.sha256(chunk['content'].encode('utf-8')).hexdigest()

    # Quick pre-check to show skip/process status
    if db.check_hash(chunk_id, source_hash):
        skipped += 1
        rprint(f'  [dim]⏭  {chunk_id} — unchanged, skipping[/dim]')
        continue

    rprint(f'  [blue]🔄 {chunk_id}[/blue] — [bold]{chunk["title"]}[/bold]')
    try:
        result = graph.invoke({
            'chunk_id': chunk_id,
            'source_content': chunk['content'],
            'source_hash': source_hash,
            'force_refresh': False,
            'revision_count': 0,
            'artifact': None,
            'skip_processing': False,
        })
        processed += 1
        artifact = result.get('artifact')
        if artifact:
            rprint(f'    ✓ {len(artifact.qa_pairs)} Q&A pairs, revisions: {result.get("revision_count", 0)}')
    except Exception as e:
        errors.append((chunk_id, str(e)))
        rprint(f'    [red]❌ {e}[/red]')

rprint()
rprint(f'[green]Done.[/green]  Processed: [bold]{processed}[/bold]  |  Skipped: [bold]{skipped}[/bold]  |  Errors: [bold]{len(errors)}[/bold]  |  Total: [bold]{len(chunks)}[/bold]')

## Post-Processing Deduplication (SOLO-42)
Scan the generated Q&A pairs for near-identical questions and deduplicate.

In [ ]:
# TODO (SOLO-42): Implement deduplication logic
# from note_taker.processing import deduplicate_qa
# deduplicate_qa(db, BOOK_CHAPTER)

## Inspect Results
Query the development SQLite database and display all generated artifacts.

In [ ]:
results_table = Table(title='Generated Artifacts')
results_table.add_column('Chunk ID', style='cyan')
results_table.add_column('Outline', justify='right')
results_table.add_column('Q&A Pairs', justify='right', style='green')
results_table.add_column('Sample Question', max_width=50)

total_qa = 0
for idx in range(len(chunks)):
    chunk_id = f'{BOOK_CHAPTER}:{idx:03d}'
    artifact = db.get_artifact(chunk_id)
    if artifact:
        total_qa += len(artifact.qa_pairs)
        sample_q = artifact.qa_pairs[0].question[:50] + '...' if artifact.qa_pairs else 'N/A'
        results_table.add_row(
            chunk_id,
            str(len(artifact.outline)),
            str(len(artifact.qa_pairs)),
            sample_q
        )
    else:
        results_table.add_row(chunk_id, '-', '-', '[dim]not processed[/dim]')

rprint(results_table)
rprint(f'\n[bold]Total Q&A pairs generated: {total_qa}[/bold]')